# N-Grams

Experimenting with different methodologies on utilizing both the infinigram API and GPT2, as well as Pythia/OLMO checkpoints.

In [17]:
# ### QUICK TEST
#
# from transformer_lens import HookedTransformer
#
# # load a tiny model (70M params)
# model = HookedTransformer.from_pretrained("EleutherAI/pythia-70m")
# tok = model.tokenizer
#
# # test it
# tokens = tok("Hello world, this is a test", return_tensors="pt")
# out = model.run_with_cache(tokens.input_ids)
# # out["logits"] etc. are now available for your n-gram / activation dumps

In [4]:
# ! aws s3 cp --no-sign-request --recursive s3://infini-gram-lite/index/v4_pileval_llama data/corpus/v4_pileval_gpt2

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


download: s3://infini-gram-lite/index/v4_pileval_llama/offset.0 to data/corpus/v4_pileval_gpt2/offset.0
download: s3://infini-gram-lite/index/v4_pileval_llama/metaoff.0 to data/corpus/v4_pileval_gpt2/metaoff.0
download: s3://infini-gram-lite/index/v4_pileval_llama/metadata.0 to data/corpus/v4_pileval_gpt2/metadata.0
download: s3://infini-gram-lite/index/v4_pileval_llama/table.0 to data/corpus/v4_pileval_gpt2/table.0
download: s3://infini-gram-lite/index/v4_pileval_llama/tokenized.0 to data/corpus/v4_pileval_gpt2/tokenized.0


In [18]:
from transformer_lens import HookedTransformer
from infini_gram.engine import InfiniGramEngine
import torch
import collections

In [19]:
model = HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer

Loaded pretrained model gpt2 into HookedTransformer


In [20]:
engine = InfiniGramEngine(
  index_dir="./data/corpus/v4_pileval_gpt2",
  eos_token_id=tokenizer.eos_token_id
)

In [21]:
# Replace this with any longer text (list of strings) or a DataLoader
text = "In the field of machine learning, transformers have revolutionized NLP."
ids = tokenizer(text, return_tensors="pt", padding=False).input_ids[0]  # [batch, seq_len]

logits, cache = model.run_with_cache(ids.unsqueeze(0))

In [22]:
layer = 4
acts  = cache["resid_post", layer][0]

In [23]:
# find the token ID for " the"
tid = tokenizer.encode(" the", add_special_tokens=False)
print("token IDs:", tid)              # e.g. [464]

# count it
res = engine.count(input_ids=tid)
print(res)

token IDs: [262]
{'count': 511287, 'approx': False}


In [23]:
import torch

# pick a feature i
i = 0
# find its top‑5 spikes
vals, idxs = torch.topk(acts[:, i], k=5)   # vals unused
for pos in idxs.tolist():
    token_id = int(ids[pos])
    cnt      = engine.count(input_ids=[token_id])["count"]
    print(f"neuron {i} spike at pos {pos} on token", tokenizer.decode([token_id]), "→ count", cnt)


neuron 0 spike at pos 3 on token  of → count 250880
neuron 0 spike at pos 9 on token  have → count 127148
neuron 0 spike at pos 14 on token . → count 14465252
neuron 0 spike at pos 12 on token  N → count 167316
neuron 0 spike at pos 13 on token LP → count 1853


In [30]:
k = 5
d_model = acts.shape[1]

# 1a) find top‑k spike positions for all features
vals, idxs = torch.topk(acts, k, dim=0)   # idxs: [k, d_model]

# 1b) for each feature, build its windows and count
n = 1
feature_counts = {}
for feat in range(d_model):
    cnts = []
    for pos in idxs[:, feat].tolist():
        start = max(0, pos - (n-1)//2)
        end   = min(len(ids), pos + (n//2) + 1)
        window_ids = ids[start:end].tolist()      # now 3‑gram
        cnts.append(engine.count(input_ids=window_ids)["count"])
    feature_counts[feat] = cnts

In [31]:
print(feature_counts)

{0: [250880, 127148, 14465252, 167316, 1853], 1: [167316, 18813, 511287, 127148, 1853], 2: [4993, 1853, 8248, 511287, 6182], 3: [14376, 70, 6182, 4993, 511287], 4: [250880, 8248, 4993, 14376, 167316], 5: [14376, 8248, 127148, 13, 154225], 6: [14465252, 6182, 250880, 13, 4993], 7: [8248, 127148, 1853, 14465252, 167316], 8: [154225, 18813, 6182, 167316, 70], 9: [8248, 1853, 14376, 4993, 250880], 10: [6182, 127148, 8248, 13, 14465252], 11: [1853, 6182, 8248, 14465252, 154225], 12: [70, 14376, 18813, 127148, 8248], 13: [14376, 18813, 250880, 8248, 511287], 14: [127148, 4993, 14465252, 511287, 13], 15: [1853, 6182, 154225, 13, 167316], 16: [511287, 8248, 6182, 14465252, 1853], 17: [1853, 8248, 167316, 4993, 250880], 18: [167316, 14376, 154225, 70, 1853], 19: [6182, 127148, 511287, 167316, 13], 20: [250880, 8248, 127148, 6182, 14376], 21: [14376, 6182, 1853, 8248, 4993], 22: [4993, 70, 18813, 250880, 14376], 23: [8248, 6182, 154225, 13, 127148], 24: [511287, 1853, 6182, 127148, 167316], 25: 

In [32]:
import numpy as np

feature_summary = {}
for feat, cnts in feature_counts.items():
    feature_summary[feat] = {
        "max_count":   max(cnts),
        "mean_count":  np.mean(cnts),
        "med_count":   np.median(cnts),
    }

In [33]:
for feat, summary in feature_summary.items():
    summary["log1p_max"]  = np.log1p(summary["max_count"])
    summary["log1p_mean"] = np.log1p(summary["mean_count"])

In [34]:
import pandas as pd

df = pd.DataFrame.from_dict(feature_summary, orient="index")
df.index.name = "feature"
df.reset_index(inplace=True)
df.to_csv("layer4_1gram_freq_summary.csv", index=False)

In [35]:
print(df)

     feature  max_count  mean_count  med_count  log1p_max  log1p_mean
0          0   14465252   3002489.8   167316.0  16.487260   14.914953
1          1     511287    165283.4   127148.0  13.144688   12.015423
2          2     511287    106512.6     6182.0  13.144688   11.576028
3          3     511287    107381.6     6182.0  13.144688   11.584153
4          4     250880     89162.6    14376.0  12.432734   11.398228
..       ...        ...         ...        ...        ...         ...
763      763     511287    162308.6   127148.0  13.144688   11.997261
764      764     511287    158216.2    18813.0  13.144688   11.971724
765      765     250880     58477.4    14376.0  12.432734   10.976413
766      766     511287    129721.2     8248.0  13.144688   11.773151
767      767   14465252   2968672.6   127148.0  16.487260   14.903626

[768 rows x 6 columns]
